Prepare our storage

In [0]:
# ShopSphere project configuration

catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_name = "shopsphere"

# Create project schema
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema_name}`"
)

# Create raw source volume
spark.sql(
    f"""CREATE VOLUME IF NOT EXISTS
    `{catalog}`.`{schema_name}`.`raw_source`"""
)

BASE_PATH = f"/Volumes/{catalog}/{schema_name}/raw_source"

print("Catalog:", catalog)
print("Schema:", schema_name)
print("Storage:", BASE_PATH)

Import libraries

In [0]:
import random
from datetime import datetime, timedelta

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

random.seed(42)

print("Libraries imported successfully")

Generate customers and products

In [0]:
names = [
    "Rahul Sharma", "Priya Verma", "Amit Singh",
    "Neha Gupta", "Rohit Kumar", "Ananya Rao",
    "Karan Mehta", "Sneha Iyer", "Arjun Patel",
    "Riya Kapoor", "Vikram Malhotra", "Ishita Nair",
    "Aditya Reddy", "Pooja Mishra", "Varun Khanna",
    "Kavya Menon", "Siddharth Jain", "Meera Joshi",
    "Akash Agarwal", "Nisha Bhat", "Manish Sinha",
    "Divya Shetty", "Harsh Vardhan", "Tanvi Desai",
    "Nikhil Chawla", "Shreya Kulkarni", "Ankit Yadav",
    "Aarav Shah", "Aditi Saxena", "Rohan Das",
    "Simran Kaur", "Abhishek Tiwari", "Maya Thomas",
    "Saurabh Pandey", "Ira Mukherjee", "Kabir Bose",
    "Diya Fernandes", "Varsha Pillai", "Mohit Arora",
    "Sanjana Rao", "Dev Malhotra", "Avni Ghosh",
    "Pranav Kulkarni", "Ritika Sen", "Yash Bansal",
    "Anushka Roy", "Gaurav Dubey", "Tara Krishnan",
    "Rishabh Saxena", "Mitali Banerjee"
]

cities = [
    "Bangalore", "Mumbai", "Delhi",
    "Hyderabad", "Pune", "Chennai",
    "Kolkata", "Jaipur", "Ahmedabad", "Surat",
    "Lucknow", "Nagpur", "Patna",
    "Ghaziabad", "Noida", "Faridabad", "Gurgaon",
    "Bhopal", "Indore", "Vadodara", "Ludhiana"
]

regions = [
    "South", "North", "West", "East"
]

categories = [
    "Electronics", "Fashion",
    "Home", "Books", "Sports"
]

# Customers
customers = []

for i in range(1, 1001):

    name = random.choice(names)

    city = random.choice(cities)

    if i % 137 == 0:
        name = None

    if i % 149 == 0:
        city = None

    customers.append((
        f"C{i:05d}",
        name,
        f"customer{i}@example.com",
        city,
        str(datetime(2024, 1, 1) + timedelta(days=random.randint(0, 700)))[:10],
        random.choice(regions)
    ))

# Products
products = []

for i in range(1, 101):

    products.append((
        f"P{i:03d}",
        f"Product_{i}",
        random.choice(categories),
        str(round(random.uniform(10, 500), 2)),
        "TRUE"
    ))

print("Customers generated:", len(customers))
print("Products generated:", len(products))

Generate orders, order items and payments

In [0]:
orders = []
order_items = []
payments = []

order_statuses = [
    "COMPLETED",
    "PENDING",
    "CANCELLED",
    "SHIPPED"
]

payment_methods = [
    "UPI", "CARD", "NET_BANKING", "COD"
]

product_lookup = {
    product[0]: float(product[3])
    for product in products
}

for i in range(1, 5001):

    order_id = f"O{i:06d}"

    customer_id = f"C{random.randint(1, 1000):05d}"

    order_date = (
        datetime(2025, 1, 1)
        + timedelta(days=random.randint(0, 365))
    ).strftime("%Y-%m-%d")

    status = random.choice(order_statuses)

    orders.append((
        order_id,
        customer_id,
        order_date,
        status,
        random.choice(["WEB", "MOBILE", "STORE"])
    ))

    total_amount = 0

    # Two products per order
    selected_products = random.sample(
        list(product_lookup.keys()), 2
    )

    for j, product_id in enumerate(selected_products):

        quantity = random.randint(1, 5)

        unit_price = product_lookup[product_id]

        total_amount += quantity * unit_price

        order_items.append((
            f"I{i:06d}{j+1}",
            order_id,
            product_id,
            str(quantity),
            str(round(unit_price, 2))
        ))

    payments.append((
        f"PAY{i:06d}",
        order_id,
        order_date,
        random.choice(payment_methods),
        "SUCCESS" if status != "CANCELLED" else "FAILED",
        str(round(total_amount, 2))
    ))

print("Orders:", len(orders))
print("Order items:", len(order_items))
print("Payments:", len(payments))

Introduce realistic data-quality issues

In [0]:
# Introduce intentional data-quality issues

# Duplicate customer records
customers.extend(customers[:20])

# Duplicate order records
orders.extend(orders[:25])

# Duplicate order items
order_items.extend(order_items[:30])

# Duplicate payments
payments.extend(payments[:20])

# Invalid product price
products[4] = (
    products[4][0],
    products[4][1],
    products[4][2],
    "-25.00",
    products[4][4]
)

# Invalid order item quantity
order_items[0] = (
    order_items[0][0],
    order_items[0][1],
    order_items[0][2],
    "0",
    order_items[0][4]
)

# Invalid payment amount
payments[0] = (
    payments[0][0],
    payments[0][1],
    payments[0][2],
    payments[0][3],
    payments[0][4],
    "INVALID"
)

print("Data quality issues introduced.")

Convert Python records into Spark DataFrames

In [0]:
def create_source_df(rows, columns):

    schema = StructType([
        StructField(col, StringType(), True)
        for col in columns
    ])

    return spark.createDataFrame(rows, schema)


customers_df = create_source_df(
    customers,
    [
        "customer_id",
        "customer_name",
        "email",
        "city",
        "signup_date",
        "region"
    ]
)

products_df = create_source_df(
    products,
    [
        "product_id",
        "product_name",
        "category",
        "unit_price",
        "active_flag"
    ]
)

orders_df = create_source_df(
    orders,
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
        "sales_channel"
    ]
)

order_items_df = create_source_df(
    order_items,
    [
        "item_id",
        "order_id",
        "product_id",
        "quantity",
        "unit_price"
    ]
)

payments_df = create_source_df(
    payments,
    [
        "payment_id",
        "order_id",
        "payment_date",
        "payment_method",
        "payment_status",
        "amount"
    ]
)

print("All five Spark DataFrames created.")

Write the raw CSV files

In [0]:
source_datasets = {
    "customers": customers_df,
    "products": products_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "payments": payments_df
}

for name, dataframe in source_datasets.items():

    output_path = f"{BASE_PATH}/{name}"

    (
        dataframe
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(output_path)
    )

    print(f"{name} successfully saved.")

print("All source datasets generated!")

Verify our datasets

In [0]:
for name in source_datasets.keys():

    path = f"{BASE_PATH}/{name}"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(path)
    )

    print(f"\nDataset: {name}")
    print("Record count:", df.count())

    df.show(5, truncate=False)